# `ptof_obs_latency_detection`

## What this notebook does
Detects two distinct failure classes at the "is the agent responsive and reliable" layer:
**responsiveness regressions** (a capability got slower than its own historical baseline) and
**sustained failure** (a capability/model_config combination is failing hard for hours, not just
having one bad minute). Also produces several WARN-tier / pipeline-health tables that are
console-visible but not part of v1's Teams-notify surface.

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `02_latency_detection` -- runs in parallel with
  `03_malformed_output`, `04_hallucination_detection`, `05_behavioral_correlation`, after
  `01_bronze_projections`, before `06_alert`.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_latency_baseline` (built nightly by `ptof_obs_nightly_baseline`), and
  `capability_registry` (human-curated by `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `latency_anomaly_findings` (v1 detector
  `latency_anomaly`, CRITICAL) and `capability_error_rate_findings` (v1 detector
  `capability_error_rate_sustained`, CRITICAL). The other tables here
  (`credential_fastfail_daily`, `write_lag_daily`, `latency_failures`, `capability_health`,
  `capability_silence`, `shift_context_missing`) are pipeline/infra-health signals, kept
  WARN-visible in task output but deliberately excluded from v1's notify surface -- they answer
  "is the pipeline healthy," not "did the agent behave."

## Tables/views touched
- **Reads:** `v_llm_bronze`, `capability_latency_baseline`, `capability_registry`,
  `capability_health` (this notebook's own output, read back in a later cell).
- **Writes:** `latency_anomalies` (raw per-call anomaly rows), `latency_anomaly_findings`
  (aggregated, signature-keyed -- what `ptof_obs_alert` actually reads),
  `credential_fastfail_daily`, `write_lag_daily`, `latency_failures`, `capability_health`,
  `capability_silence`, `shift_context_missing`, `capability_error_rate_alert` (window-aggregated
  sustained-failure detector), `capability_error_rate_findings` (signature-keyed version of the
  same, what `ptof_obs_alert` reads).


In [ ]:
%sql
-- latency_anomalies -- findings only, reliability-aware. Per-call detail table: every individual
-- call this run classified as slower than "ok" for its capability, not yet aggregated into an
-- incident. latency_anomaly_findings (next cell) is what actually feeds ptof_obs_alert.
-- Explicitly JOINs capability_registry WHERE active = true for consistency with every other
-- detector — prevents a deactivated capability with a stale baseline from producing anomaly
-- findings for up to 30 days.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.latency_anomalies AS
SELECT * FROM (
  SELECT
      b.id, b.shift_date, b.shift_type, b.batch_nbr,
      b.capability, b.model_config, b.transport, b.scheduler_run,
      b.latency_ms, b.user_prompt_chars,
      base.p95_ms, base.p99_ms, base.anomaly_upper_bound_ms,
      base.n_samples, base.is_reliable,
      -- latency_verdict: classifies this call against ITS OWN capability's baseline, not a fixed
      -- SLA -- a naturally slower capability shouldn't trip the same bar as a fast one. Falls back
      -- to a self-scaling ceiling (3x this capability's own thin p99, floor 10s) when there's no
      -- reliable baseline yet. model_config is kept for attribution, not for baseline comparison.
      CASE
        WHEN base.capability IS NULL                                             THEN 'no_baseline'
        WHEN base.is_reliable = false
             AND b.latency_ms > GREATEST(3 * base.p99_ms, 10000)                  THEN 'anomaly_fixed_ceiling'
        WHEN base.is_reliable = false                            THEN 'baseline_unreliable'
        WHEN b.latency_ms > base.anomaly_upper_bound_ms           THEN 'anomaly'
        WHEN b.latency_ms > base.p99_ms                          THEN 'p99_breach'
        WHEN b.latency_ms > base.p95_ms                          THEN 'p95_breach'
        ELSE 'ok'
      END AS latency_verdict,
      b.called_at,
      current_timestamp() AS detected_at
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LEFT JOIN mq_gmdf_dev.oil_obs.capability_latency_baseline base
         ON base.capability = b.capability
  WHERE b.called_at >= current_timestamp() - INTERVAL 60 MINUTES
    AND b.is_credential_fastfail = false
    AND b.success                = true
)
WHERE latency_verdict <> 'ok';

In [ ]:
%sql
-- latency_anomaly_findings -- aggregated per (capability, verdict) for MERGE. This is what
-- ptof_obs_alert.ipynb's latency_anomaly detector (CRITICAL in v1) actually reads.
-- Keyed on the CONDITION (capability + verdict), not per-call id, so a persistently slow
-- capability is ONE incident whose detection_count climbs.
-- model_config collected as attribution (which configs were involved), not as a grouping key --
-- per SME, model_config has minimal effect on SAA/ISH agents.
-- HAVING count(*) >= 2: requires at least 2 anomalous calls in the window before producing a
-- finding — a single slow call is noise at 5-15 calls/hour volume.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.latency_anomaly_findings AS
SELECT
    capability,
    collect_set(model_config) AS model_configs_involved,
    latency_verdict,
    max(latency_ms) AS worst_latency_ms,
    count(*) AS anomaly_count,
    max(p95_ms) AS baseline_p95_ms,
    max(anomaly_upper_bound_ms) AS anomaly_bound_ms,
    max(called_at) AS latest_called_at,
    -- finding_signature: the dedup key ptof_obs_alert's obs_incidents MERGE keys on. Derived
    -- from (capability, verdict) only -- model_config is no longer part of the signature.
    -- Deliberate dedup-history reset: old per-config signatures won't match new ones.
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(latency_verdict, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.latency_anomalies
WHERE latency_verdict IN ('anomaly', 'anomaly_fixed_ceiling')
GROUP BY capability, latency_verdict
HAVING count(*) >= 2;

In [ ]:
%sql
-- credential_fastfail_daily — WARN-tier pipeline-health signal, not part of v1's Teams-notify
-- surface. Tracks the specific known failure mode (misconfigured demo model hitting "Unable to
-- locate credentials" immediately) so a spike reads as a config problem, not generic latency.
-- Time-bounded to 30 days: nobody triages fastfails from months ago, and the scan grows
-- linearly with total data volume without this bound.
-- Scoped to active capabilities via capability_registry join so deactivated DSA/probe
-- capabilities don't inflate fastfail counts.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.credential_fastfail_daily AS
SELECT
    date_trunc('DAY', b.called_at) AS day,
    b.model_config,
    count(*)                     AS fastfail_calls,
    approx_percentile(b.latency_ms, array(0.50, 0.95)) AS fastfail_latency_pcts,
    approx_count_distinct(concat_ws('|', b.shift_date, b.shift_type, b.batch_nbr)) AS batches_impacted
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.is_credential_fastfail = true
  AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
GROUP BY 1, 2;

In [0]:
%sql
-- write_lag_daily — WARN-tier pipeline-health signal (is the ingestion pipeline itself keeping
-- up), not agent-behavior quality. Not part of v1's Teams-notify surface.
-- write_lag_s spans the whole call, so a 70 s generation registers as 73 s of "ingestion lag".
-- Subtracting latency_ms isolates the time between the call returning and the row landing.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.write_lag_daily AS
SELECT
    date_trunc('HOUR', ingestion_ts) AS hour,
    approx_percentile(write_lag_s, 0.50) AS p50_s,
    approx_percentile(write_lag_s, 0.95) AS p95_s,
    approx_percentile(write_lag_s, 0.99) AS p99_s,
    -- p95_ingest_only_s: the pure ingestion-pipeline lag, with call duration subtracted out --
    -- the number that matters if you're asking "is the write path itself slow," as opposed to
    -- "was the LLM call slow."
    approx_percentile(write_lag_s - latency_ms/1000.0, 0.95) AS p95_ingest_only_s,
    count_if(write_lag_s - latency_ms/1000.0 > 60)           AS ingest_over_60s_rows,
    count_if(write_lag_s > 60)                               AS lag_over_60s_rows,
    count(*)                                                 AS total_rows
FROM mq_gmdf_dev.oil_obs.v_llm_bronze
WHERE ingestion_ts >= current_timestamp() - INTERVAL 7 DAYS
  AND success = true
GROUP BY 1;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- latency_failures — raw per-hour error_class breakdown. Excluded from v1 (per the implementation
-- plan): no threshold/findings table built on top yet, needs real design work rather than a
-- wiring fix. Kept here as a console-visible diagnostic breakdown by error taxonomy.
-- Scoped to active capabilities via capability_registry join so deactivated DSA/probe
-- failures don't inflate error breakdowns.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.latency_failures AS
SELECT
    date_trunc('HOUR', b.called_at) AS hour,
    b.capability, b.model_config, b.scheduler_run, b.error_class,
    count(*)                              AS failed_calls,
    approx_percentile(b.latency_ms, 0.50) AS p50_ms,
    max(b.latency_ms)                     AS max_ms,
    max(b.user_prompt_chars)              AS max_prompt_chars
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.success = false
  AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY 1,2,3,4,5;

In [0]:
%sql
-- capability_health — error rate per capability per hour. The per-hour version is WARN-only in
-- v1 (a single bad hour can be noise); capability_error_rate_alert below aggregates this over a
-- 6h window to catch SUSTAINED failure, which is what actually reaches Teams.
-- Deliberately does NOT filter success = true. This is the detector for dsa_optimize.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_health AS
SELECT
    date_trunc('HOUR', b.called_at)               AS hour,
    b.capability,
    b.model_config,
    count(*)                                      AS total_calls,
    count_if(b.success)                           AS success_calls,
    count_if(NOT b.success)                       AS failed_calls,
    count_if(NOT b.success) * 1.0 / count(*)      AS error_rate,
    concat_ws(' | ', collect_set(b.error_class))  AS error_classes,
    concat_ws(' | ', collect_set(nullif(b.error_msg, ''))) AS error_msgs,
    approx_percentile(b.latency_ms, 0.95)         AS p95_ms,
    max(b.user_prompt_chars)                      AS max_prompt_chars
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY 1, 2, 3; 

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- capability_silence — WARN-tier pipeline-health signal: is a capability that should be calling
-- regularly still calling at all. Not part of v1's notify surface (ops health, not agent
-- behavior).
-- capability_silence — time since last call, not daily count.
-- dsa_copilot is human-driven with a ~10h overnight gap, so a daily-count threshold
-- false-fires every morning as the trailing window rolls past the evening burst.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence AS
SELECT
    r.capability, r.expected_min_daily, r.silence_grace_hours, r.owner,
    count(b.id)      AS calls_last_7d,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
      AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
WHERE r.active = true AND r.silence_grace_hours IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_grace_hours * 3600;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- shift_context_missing — WARN-tier standing gap, not part of v1's notify surface (it's an
-- upstream data-completeness issue, not an agent-behavior failure, and it's permanent until the
-- dsa_* capabilities start populating shift/batch identity).
-- shift_context_missing — the dsa_* capabilities write shift_date=NULL, shift_type='' and
-- batch_nbr='' on every row, which blocks all AI-to-ISH correlation and makes any per-shift or
-- per-batch slice impossible. June's summary capability populated all three, so the schema
-- supports it — the DSA surface just isn't filling them in.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.shift_context_missing AS
SELECT
    b.capability,
    count(*)                                                 AS total_calls,
    count_if(coalesce(b.shift_type, '') = '')                AS blank_shift_type,
    count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') AS blank_batch_nbr,
    count_if(b.shift_date IS NULL)                           AS null_shift_date,
    max(b.called_at)                                         AS last_seen,
    current_timestamp()                                      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY b.capability
HAVING count_if(coalesce(b.shift_type, '') = '') > 0
    OR count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') > 0
    OR count_if(b.shift_date IS NULL) > 0;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- capability_error_rate_alert — the "is the agent actually working" detector, sustained over a
-- 6h window so a single quiet/bad hour can't hide (or fake) an outage. Feeds
-- capability_error_rate_findings below, which is what ptof_obs_alert's
-- capability_error_rate_sustained detector (CRITICAL in v1) actually reads.
-- capability_error_rate_alert — window-aggregated, so a sustained outage cannot hide behind one
-- quiet hour. Applies the volume floor to the WINDOW total, not to any single hour.
-- Threshold lowered from 0.8 to 0.5 for GxP-only fleet (SAA/ISH rescope): 50% = majority-
-- failing gate. Observed data is bimodal (0% or 93%+), so no noise increase at this level.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_error_rate_alert AS
SELECT
    capability, model_config,
    sum(total_calls)  AS calls_6h,
    sum(failed_calls) AS failures_6h,
    round(sum(failed_calls) * 1.0 / nullif(sum(total_calls), 0), 3) AS error_rate_6h,
    count(*)                   AS hours_with_data,
    count_if(error_rate = 1.0) AS hours_fully_failed,
    max(hour)                  AS latest_hour
FROM mq_gmdf_dev.oil_obs.capability_health
WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL 6 HOURS)
GROUP BY capability, model_config
HAVING sum(total_calls) >= 5
   AND sum(failed_calls) * 1.0 / nullif(sum(total_calls), 0) >= 0.5;

In [ ]:
%sql
-- capability_error_rate_findings — signature-keyed for obs_incidents MERGE.
-- Reuses capability_error_rate_alert which already applies the correct thresholds:
--   sum(total_calls) >= 5 AND error_rate_6h >= 0.5 (lowered for GxP-only fleet)
-- aggregated over the 6h WINDOW total (not per-hour), which is what caught the sustained
-- dsa_optimize outage that the per-hour check missed.
-- Key on (capability, model_config) only — NOT the hour — so a sustained outage is one
-- incident whose detection_count climbs, not one incident per hour.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_error_rate_findings AS
SELECT
    capability, model_config,
    calls_6h, failures_6h, error_rate_6h,
    hours_with_data, hours_fully_failed, latest_hour,
    -- finding_signature: the dedup key ptof_obs_alert's obs_incidents MERGE keys on -- same
    -- pattern as latency_anomaly_findings above, so a sustained outage stays one incident.
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(model_config, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.capability_error_rate_alert;

In [ ]:
%sql
-- capability_outage_findings — capability-level outage rollup. Where
-- capability_error_rate_sustained groups by (capability, model_config), this groups by
-- capability only: if all configs fail, it produces one CRITICAL incident instead of N
-- separate Teams cards. A cleaner triage experience during total outages.
-- Uses the same 6h window and 50% threshold as capability_error_rate_alert for consistency.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_outage_findings AS
SELECT capability,
       sum(total_calls) AS calls_6h,
       sum(failed_calls) * 1.0 / nullif(sum(total_calls), 0) AS error_rate_6h,
       collect_set(model_config) AS model_configs_affected,
       count(DISTINCT model_config) AS configs_affected,
       sha2(capability, 256) AS finding_signature,
       current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.capability_health
WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL 6 HOURS)
GROUP BY capability
HAVING sum(total_calls) >= 5
   AND sum(failed_calls) * 1.0 / nullif(sum(total_calls), 0) >= 0.5;

In [ ]:
%sql
-- prompt_size_drift — detects when a capability's prompt or response size in the last hour
-- exceeds 2x its nightly baseline p95. A prompt template regression (accidentally embedding
-- full shift history, wrong model config, copy-paste error) silently increases cost and latency.
-- This is a leading indicator for latency anomalies — it fires before the latency threshold
-- does, giving time to catch a deployment regression within the hour.
-- Requires p95_prompt_chars and p95_response_chars in capability_latency_baseline (Phase 6b).
-- Scoped to active capabilities via capability_registry join: prevents stale baselines for
-- deactivated capabilities (which persist up to 30 days) from producing false drift findings.
-- finding_signature keyed on capability only (not model_config): per SME, prompt templates are
-- capability-level, not config-level, so a size regression is one incident per capability.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.prompt_size_drift AS
SELECT b.capability,
       percentile_approx(b.user_prompt_chars, 0.5) AS median_prompt_chars_1h,
       max(base.p95_prompt_chars) AS baseline_p95_prompt_chars,
       percentile_approx(b.response_chars, 0.5) AS median_response_chars_1h,
       max(base.p95_response_chars) AS baseline_p95_response_chars,
       count(*) AS calls_in_window,
       sha2(b.capability, 256) AS finding_signature,
       current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
JOIN mq_gmdf_dev.oil_obs.capability_latency_baseline base ON base.capability = b.capability
WHERE b.called_at >= current_timestamp() - INTERVAL 60 MINUTES
  AND b.success = true
GROUP BY b.capability
HAVING percentile_approx(b.user_prompt_chars, 0.5) > 2 * max(base.p95_prompt_chars)
    OR percentile_approx(b.response_chars, 0.5) > 2 * max(base.p95_response_chars);